# MLP Knowledge Storage

What knowledge is stored in MLP weights: neuron logit effects,
layer effects, knowledge localization, and association structure.

## Why This Matters

Knowledge mlp storage investigates where and how models store factual information. Locating knowledge in specific components (neurons, layers, directions) is essential for understanding hallucination, enabling fact editing, and assessing what a model truly knows.

**Key references:**
- [Geva et al. (2021) "Transformer Feed-Forward Layers Are Key-Value Memories"](https://arxiv.org/abs/2012.14913) — Shows MLPs function as key-value memory stores
- [Meng et al. (2022) "Locating and Editing Factual Associations"](https://arxiv.org/abs/2202.05262) — Causal tracing to locate factual knowledge

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.mlp_knowledge_storage import (
    neuron_logit_effect, mlp_layer_logit_effect,
    knowledge_localization, mlp_association_structure,
    mlp_knowledge_summary,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Neuron Logit Effects

What tokens does each neuron promote or suppress?

In [ ]:
for layer in range(4):
    result = neuron_logit_effect(model, layer=layer, top_k=3)
    print(f"  Layer {layer}:")
    for p in result['per_neuron']:
        promoted = [(t, f'{l:.3f}') for t, l in p['promoted']]
        print(f"    N{p['neuron']}: promotes={promoted}, range={p['effect_range']:.4f}")

## Layer-Level Logit Effect

In [ ]:
for layer in range(4):
    result = mlp_layer_logit_effect(model, tokens, layer=layer, position=-1, top_k=3)
    promoted = [(t, f'{l:.3f}') for t, l in result['promoted']]
    print(f"  Layer {layer}: promotes={promoted}, norm={result['total_effect_norm']:.4f}")

## Knowledge Localization

Which layer contributes most to a specific token prediction?

In [ ]:
for target in [1, 15, 30]:
    result = knowledge_localization(model, tokens, target_token=target, position=-1)
    print(f"  Token {target}: most important layer={result['most_important_layer']}")
    for p in result['per_layer']:
        bar = '█' * int(p['fraction'] * 30)
        print(f"    L{p['layer']}: contrib={p['contribution']:+.4f}, frac={p['fraction']:.3f} {bar}")

## Association Structure

In [ ]:
for layer in range(4):
    result = mlp_association_structure(model, layer=layer, top_k=3)
    top_sv = [f'{v:.4f}' for v in result['top_singular_values']]
    print(f"  Layer {layer}: eff_rank={result['effective_rank']:.2f}, "
          f"cond={result['condition_number']:.1f}, top_sv={top_sv}")

## Summary

In [ ]:
result = mlp_knowledge_summary(model, tokens, position=-1)
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: top_token={p['top_promoted_token']}, "
          f"norm={p['effect_norm']:.4f}, rank={p['effective_rank']:.2f}")